In [ ]:
from Tools_U1_plaquette import *

In [ ]:
# ---------------- Hamiltonian constants ----------------
m = 5.0
g = 0.2
λ = 1.0

ECm = m / 4
ECg = g / 4
EJ = λ

# ---------------- Define the size of each local Hilbert space ----------------
Nsize = 11 # 9 for the long-time plot, 11 for the short-time plot

# ---------------- Time evolution parameters ----------------
dt = 1.0
t_max = 3500.0 # 10000.0 for the long-time plot, 3500.0 for the short-time plot
times = np.arange(0,t_max + dt,dt)

# ---------------- Hamiltonian in charge basis {(nθ12, nθ24, nθ13, nθ34)} (! this order) ----------------
JJcapacitance = False

if JJcapacitance:
    χ = 0.1
    EC_mg = χ / 4
    η = 1.0
    EC_mm = η / 4
    H = inf_space_LGT_plaquette_charge_basis_Matrix_reduced4vars_alpha1minus1(
        EC_m1 = ECm, EC_g1 = ECg, EJ1 = EJ, 
        EC_m2 = ECm, EC_g2 = ECg, EJ2 = EJ, 
        EC_m3 = ECm, EC_g3 = ECg, EJ3 = EJ, 
        EC_m4 = ECm, EC_g4 = ECg, EJ4 = EJ, 
        Nsize = Nsize,
        EC_mg1 = EC_mg, EC_mm1 = EC_mm,
        EC_mg2 = EC_mg, EC_mm2 = EC_mm,
        EC_mg3 = EC_mg, EC_mm3 = EC_mm, 
        EC_mg4 = EC_mg, EC_mm4 = EC_mm
    )
else:
    H = inf_space_LGT_plaquette_charge_basis_Matrix_reduced4vars_alpha1minus1(
        EC_m1 = ECm, EC_g1 = ECg, EJ1 = EJ, 
        EC_m2 = ECm, EC_g2 = ECg, EJ2 = EJ, 
        EC_m3 = ECm, EC_g3 = ECg, EJ3 = EJ, 
        EC_m4 = ECm, EC_g4 = ECg, EJ4 = EJ, 
        Nsize = Nsize
    )
print(H.shape)

# ---------------- Define the initial states UP/DOWN ----------------
ket_one_N_1 = vector_sparse_ndim_1idx(Nsize, [Nsize// 2 +1])
ket_one_N_0 = vector_sparse_ndim_1idx(Nsize, [Nsize //2])
print(ket_one_N_1.toarray())
print(ket_one_N_0.toarray())

ψ_0_UP = kron_n(ket_one_N_1 ,ket_one_N_1, ket_one_N_0, ket_one_N_0 ) 
ψ_0_UP /= sp.linalg.norm(ψ_0_UP)

ψ_0_DOWN = kron_n(ket_one_N_0 ,ket_one_N_0, ket_one_N_1, ket_one_N_1 )
ψ_0_DOWN /= sp.linalg.norm(ψ_0_DOWN)

# ---------------- Define the time-evolved operators ----------------
M = (Nsize-1) // 2 
id_c = sp.eye(2 * M + 1)
n_c = sp.spdiags(np.arange(-M, M + 1), [0], 2 * M + 1, 2 * M + 1)
nθ12 = kron_n(n_c, id_c, id_c, id_c)
nθ24 = kron_n(id_c, n_c, id_c, id_c)
nθ13 = kron_n(id_c, id_c, n_c, id_c)
nθ34 = kron_n(id_c, id_c, id_c, n_c)
id_N = kron_n(id_c, id_c, id_c, id_c)
α_1 = -1
α_2 = 0
α_3 = 0
α_4 = 1
nϕ1 = α_1 * id_N + nθ12 + nθ13
nϕ2 = α_2 * id_N + nθ24 - nθ12
nϕ3 = α_3 * id_N + nθ34 - nθ13
nϕ4 = α_4 * id_N - nθ24 - nθ34

In [ ]:
# ---------------- Useful functions ----------------
def expectation(psi, O):
    return (psi.conj().T @ (O @ psi)).toarray()[0, 0].real

def fidelity(psi, psi_ref):
    return np.abs(psi.getH().dot(psi_ref)[0, 0])**2

# ---------------- Define the lists to fill ----------------
exp_values_nϕ1  = []
exp_values_nϕ2  = []
exp_values_nϕ3  = []
exp_values_nϕ4  = []
exp_values_nθ12  = []
exp_values_nθ24  = []
exp_values_nθ13  = []
exp_values_nθ34  = []
UPfidelity = []
DOWNfidelity = []

# ---------------- Time evolution ----------------
UP = True # Set to True for the UP initial state, False for the DOWN initial state

psi = ψ_0_UP.copy() if UP else ψ_0_DOWN.copy()

for i,t in enumerate(times):
    print(f"i = {i}, times = {t}")

    ti = time.perf_counter() 

    if i > 0:
        psi = expm_multiply(-1j * dt * H, psi)
        psi /= sp.linalg.norm(psi)

    UPfidelity.append(fidelity(psi, ψ_0_UP))
    DOWNfidelity.append(fidelity(psi, ψ_0_DOWN))

    exp_values_nϕ1.append(expectation(psi, nϕ1))
    exp_values_nϕ2.append(expectation(psi, nϕ2))
    exp_values_nϕ3.append(expectation(psi, nϕ3))
    exp_values_nϕ4.append(expectation(psi, nϕ4))
    exp_values_nθ12.append(expectation(psi, nθ12))
    exp_values_nθ24.append(expectation(psi, nθ24))
    exp_values_nθ13.append(expectation(psi, nθ13))
    exp_values_nθ34.append(expectation(psi, nθ34))

    tif = time.perf_counter() - ti
    print(f"Done in {tif:.4f} s")

In [ ]:
# ---------------- Save results ----------------
if JJcapacitance:
    filename_oscillations = f"LGT_SCQs_plaquette_reduced_def_OCT25/Oscillations_updown_reduced4varsplaquette_symbasis_alpha1minus1_with_JJcapacitance_ns_and_fidelities_vs_time_t_{t_max}_m_{m}_g_{g}_λ_{λ}_Nsize_{Nsize}_draft_def.npz"
else:
    filename_oscillations = f"LGT_SCQs_plaquette_reduced_def_OCT25/Oscillations_updown_reduced4varsplaquette_symbasis_alpha1minus1_ns_and_fidelities_vs_time_t_{t_max}_m_{m}_g_{g}_λ_{λ}_Nsize_{Nsize}_draft_def.npz"

if UP: 
    np.savez_compressed(
        filename_oscillations,
        m = m,
        g = g,
        λ = λ,
        Nsize = Nsize,
        t = times,
        expvals_nphi1_UP = exp_values_nϕ1, 
        expvals_ntheta12_UP = exp_values_nθ12, 
        expvals_nphi2_UP = exp_values_nϕ2, 
        expvals_ntheta24_UP = exp_values_nθ24, 
        expvals_nphi4_UP = exp_values_nϕ4, 
        expvals_ntheta34_UP = exp_values_nθ34,
        expvals_nphi3_UP = exp_values_nϕ3, 
        expvals_ntheta13_UP = exp_values_nθ13,
        UP_f_UP = UPfidelity,
        DOWN_f_UP = DOWNfidelity
    )
else:
    np.savez_compressed(
        filename_oscillations,
        m = m,
        g = g,
        λ = λ,
        Nsize = Nsize,
        t = times,
        expvals_nphi1_DOWN = exp_values_nϕ1, 
        expvals_ntheta12_DOWN = exp_values_nθ12, 
        expvals_nphi2_DOWN = exp_values_nϕ2, 
        expvals_ntheta24_DOWN = exp_values_nθ24, 
        expvals_nphi4_DOWN = exp_values_nϕ4, 
        expvals_ntheta34_DOWN = exp_values_nθ34,
        expvals_nphi3_DOWN = exp_values_nϕ3, 
        expvals_ntheta13_DOWN = exp_values_nθ13,
        UP_f_DOWN = UPfidelity,
        DOWN_f_DOWN = DOWNfidelity
    )
print(f"\nResults saved to {filename_oscillations}")